# Movies Medallion Architecture (Databricks Community Edition)

**Author:** *Anusha Aligi*  
**Platform:** Databricks Community Edition (Free Tier)  
**Objective:** Ingest, clean, and model Netflix and IMDB datasets using the Medallion (Bronze → Silver → Gold) pattern to support analytics.

---

## Assumptions & Environment Setup

1. **Data Source Uploads**
   - **Netflix:** Single CSV file (e.g., `netflix_titles.csv`) manually uploaded.  
   - **IMDB:** Multiple CSVs partitioned by year (e.g., `/imdb_movies/1920/*.csv` → `/imdb_movies/2025/*.csv`).  
   - Both datasets uploaded manually to:
     ```
     /Volumes/main/default/movies_data/
     ```
     Example paths:
     ```
     /Volumes/main/default/movies_data/netflix_titles.csv
     /Volumes/main/default/movies_data/imdb_movies/
     ```
2. **No Internet Access:**  
   Databricks CE doesn’t allow direct Kaggle downloads; manual upload is required.

3. **Schema Standardization:**  
   Unified fields across sources:  
   `title`, `content_type`, `release_year`, `director`, `age_certification`, `duration`, `genres`, `description`, `source_system`.

4. **Deduplication Logic:**  
   Duplicates are identified by normalized `title + release_year + source_system`.

5. **Data Cleaning:**  
   - String trimming and whitespace normalization.  
   - Safe numeric casting for runtime, year (regex extraction of digits).  
   - Missing `director` handled as nulls for Netflix.  
   - Age ratings standardized across platforms.

---

## Medallion Architecture

| Layer | Purpose | Output Tables |
|:------|:---------|:---------------|
| **Bronze** | Raw, minimally cleaned ingestions from CSVs | `netflix_bronze`, `imdb_bronze` |
| **Silver (Conformed)** | Unified curated dataset across both sources | `movies_conformed` |
| **Gold (Analytics)** | Aggregations for visualization and reporting | `titles_per_year`, `top_directors`, `age_certification_counts` |

---

## User Stories Supported

| Persona | Requirement | Dataset/Table |
|:---------|:-------------|:---------------|
| **Data Analyst** | Number of titles released per year | `titles_per_year` |
| **Content Lead** | Top directors by total content produced | `top_directors` |
| **Compliance Officer** | Most common age certifications | `age_certification_counts` |

---

## Execution Instructions

1. Upload datasets via **Workspace → Import**:
   - Netflix → `/Volumes/main/default/movies_data/netflix_titles.csv`
   - IMDB folder → `/Volumes/main/default/movies_data/imdb_movies/`
2. Update any file paths in the initial widget setup if needed.
3. Run all cells **top to bottom**:
   - Ingestion → Cleaning → Delta Table Creation  
   - Conformation → Aggregations → Visualization
4. All outputs are written to `main.default` catalog:

In [0]:
import zipfile

zip_path = "/Volumes/workspace/default/movies_data/Data.zip"
extract_dir = "/Volumes/workspace/default/movies_data/imdb_movies/"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print("Extracted IMDB CSV files to:", extract_dir)
display(dbutils.fs.ls("/Volumes/workspace/default/movies_data/imdb_movies"))

✅ Extracted IMDB CSV files to: /Volumes/workspace/default/movies_data/imdb_movies/


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/movies_data/imdb_movies/Data/,Data/,0,1763370691985


## Define paths

In [0]:
NETFLIX_CSV = "/Volumes/workspace/default/movies_data/titles.csv"
IMDB_FOLDER = "/Volumes/workspace/default/movies_data/imdb_movies/Data/"

## Collect valid IMDB CSVs (skip empty/tiny files)

In [0]:
import re

def list_csvs_nonempty(base_dir: str, min_bytes: int = 128):
    csvs = []
    for yr in dbutils.fs.ls(base_dir):
        if yr.isDir():
            for f in dbutils.fs.ls(yr.path):
                if f.name.lower().endswith(".csv") and f.size is not None and f.size >= min_bytes:
                    csvs.append(f.path)
    return csvs

imdb_csv_files = list_csvs_nonempty(IMDB_FOLDER)
print("IMDB CSV files found:", len(imdb_csv_files))
assert imdb_csv_files, "No valid IMDB CSV files found (after filtering tiny/empty files)."

IMDB CSV files found: 318


## Read Netflix & IMDB raw

In [0]:
from pyspark.sql import functions as F

netflix_raw = (
    spark.read.option("header", True).option("inferSchema", True).csv(NETFLIX_CSV)
)

imdb_raw = (
    spark.read.option("header", True).option("inferSchema", True).csv(imdb_csv_files)
)

print("Netflix cols:", netflix_raw.columns)
print("IMDB cols   :", imdb_raw.columns[:30])

Netflix cols: ['id', 'title', 'type', 'description', 'release_year', 'age_certification', 'runtime', 'genres', 'production_countries', 'seasons', 'imdb_id', 'imdb_score', 'imdb_votes', 'tmdb_popularity', 'tmdb_score']
IMDB cols   : ['Title', 'Year', 'Duration', 'MPA', 'Rating', 'Votes', 'méta_score', 'description', 'Movie Link', 'writers', 'directors', 'stars', 'budget', 'opening_weekend_Gross', 'grossWorldWWide', 'gross_US_Canada', 'release_date', 'countries_origin', 'filming_locations', 'production_company', 'awards_content', 'genres', 'Languages']


## Bronze: Netflix (mapping columns)

In [0]:
from pyspark.sql import functions as F

def clean_str(col):
    return F.trim(
        F.regexp_replace(
            F.coalesce(col, F.lit("")),
            r"\s+",
            " "
        )
    )

def to_int_safe(col):
    return F.when(
        F.regexp_extract(F.col(col).cast("string"), r'(\d+)', 1) != "",
        F.regexp_extract(F.col(col).cast("string"), r'(\d+)', 1).cast("int")
    ).otherwise(F.lit(None))

netflix_bronze = (
    netflix_raw
      .withColumn("title", clean_str(F.col("title")))
      .withColumn("director", F.lit(None).cast("string"))
      .withColumn("rating", F.upper(clean_str(F.col("age_certification"))))
      .withColumn("release_year", to_int_safe("release_year"))
      .withColumn("duration", to_int_safe("runtime"))
      .withColumn("genres", F.col("genres").cast("string"))
      .withColumn("description", clean_str(F.col("description")))
      .withColumn("content_type", F.upper(F.col("type")))
      .withColumn("norm_title", F.lower(F.regexp_replace(F.col("title"), r"[^a-zA-Z0-9]+", "")))
      .withColumn("source_system", F.lit("NETFLIX"))
      .dropDuplicates(["norm_title", "release_year", "source_system"])
)

netflix_bronze.write.format("delta").mode("overwrite").saveAsTable("workspace.default.netflix_bronze")
print("workspace.default.netflix_bronze")

✅ workspace.default.netflix_bronze


## IMDB → Bronze (safe parsing + cleaning)

In [0]:
def to_int_safe(col):
    return F.expr(f"try_cast({col} as int)")

def to_double_safe(col):
    return F.expr(f"try_cast({col} as double)")

def to_long_safe(col):
    return F.expr(f"try_cast({col} as long)")

imdb_bronze = (
    imdb_raw.select(
        clean_str(F.col("Title")).alias("title"),
        clean_str(F.col("directors")).alias("director"),
        to_int_safe("Year").alias("release_year"),
        F.upper(clean_str(F.col("MPA"))).alias("rating"),
        to_int_safe("Duration").alias("duration"),
        F.col("genres").cast("string").alias("genres"),
        clean_str(F.col("description")).alias("description"),
        to_double_safe("Rating").alias("imdb_score"),
        to_long_safe("Votes").alias("imdb_votes"),
        to_int_safe("`méta_score`").alias("meta_score"),
        clean_str(F.col("countries_origin")).alias("country")
    )
    .withColumn("content_type", F.lit("MOVIE"))
    .withColumn(
        "norm_title",
        F.lower(
            F.regexp_replace(
                F.col("title"),
                r"[^a-zA-Z0-9]+",
                ""
            )
        )
    )
    .withColumn("source_system", F.lit("IMDB"))
    .dropDuplicates(["norm_title", "release_year", "source_system"])
)

imdb_bronze.write.format("delta").mode("overwrite").saveAsTable("workspace.default.imdb_bronze")
display(imdb_bronze)

title,director,release_year,rating,duration,genres,description,imdb_score,imdb_votes,meta_score,country,content_type,norm_title,source_system
"426. Wild, Wild Susan",['A. Edward Sutherland'],1925,,null,"['Comedy', 'Romance']","Wealthy New York girl, Susan Van Dusen, in search of thrills and laughter, leaves home and finds work with a private detective agency. She meets Tod Waterbury, who, under another name, is working as a cab driver (in search of story material for a novel), and the two fall in love. Tod offers the detective agency a reward to find himself and arranges for Susan to be assigned to the case; since they are constantly together, Susan hasn't a chance in the world of finding him. Susan is assigned to another case and follows a gang of crooks to a dark and deserted house. After a series of harrowing adventures in the house, she comes to realize that the whole affair has been fabricated by Tod and her family to cure her of her lust for adventure. Susan marries Tod, greatly to her own and her father's delight.",6.1,15,null,['United States'],MOVIE,426wildwildsusan,IMDB
154. The False Road,and it turns out that the culprits are two members of Roger's old gang. He tracks them to New York and convinces them that he wants to get back into the gang,1920,,null,"April 18, 1920","""Roger Moran, a member of a gang of thieves headed by Mike Wilson, is released from prison after having served a two-year sentence. He has learned his lesson and vows to leave his life of crime, but his girlfriend Betty Palmer--also a member of the gang--won't leave """"the false road"""". Roger finally leaves her and finds a job with a sympathetic banker",6.0,41,null,,MOVIE,154thefalseroad,IMDB
41. Stranger on the Third Floor,['Boris Ingster'],1940,APPROVED,null,null,"An aspiring reporter is the key witness at the murder trial of a young man accused of cutting a café owner's throat, and is soon accused of a similar crime himself.",6.8,null,null,"August 16, 1940",MOVIE,41strangeronthethirdfloor,IMDB
236. The Ranger and the Lady,['Joseph Kane'],1940,APPROVED,null,null,"While Sam Houston in in the nation's capital trying to get Texas into the Union, his aide is trying to impose a self-serving tax on the use of the Santa Fe trail. The lady owner of a wagon train is using the trail, and a Texas Ranger comes to her assistance.",5.9,151,null,'Ted Mapes',MOVIE,236therangerandthelady,IMDB
564. Osvobozhdeniye,"['Aleksandr Dovzhenko', 'Yuliya Solntseva']",1940,,null,['Documentary'],The liberation of Ukrainian and Belorussian lands from the yoke of Polish landlords and the reunification of the sister nations into a single family.,5.3,87,null,['Soviet Union'],MOVIE,564osvobozhdeniye,IMDB
554. The Japanese Wife Next Door,['Yutaka Ikejima'],2004,NOT RATED,null,['Comedy'],A new bride convinces everyone in her family to sleep with her.,5.2,null,null,['Japan'],MOVIE,554thejapanesewifenextdoor,IMDB
203. Roman de gare,['Claude Lelouch'],2007,R,null,Alpes-Maritimes,A popular novelist researches unlikely sources to find characters for her next bestseller.,7.1,null,null,['France'],MOVIE,203romandegare,IMDB
246. Antichrist,['Lars von Trier'],2009,NOT RATED,null,"['Psychological Drama', 'Psychological Horror', 'Psychological Thriller', 'Drama', 'Horror', 'Thriller']","A grieving couple retreat to their cabin in the woods, hoping to repair their broken hearts and troubled marriage, but nature takes its course and things go from bad to worse.",6.5,null,null,"['Denmark', 'Germany', 'France', 'Sweden', 'Italy', 'Poland']",MOVIE,246antichrist,IMDB
345. Der Evangelimann,['Holger-Madsen'],1924,,null,[],"Mathias, the evangelist, is in love with Martha. Johannes, Mathias brother tries to interfere. When Mathias and Martha are swearing fidelity, Johannes jealousy turns into blind hate and he sets the monastery on fire.",null,null,null,['Germany'],MOVIE,345derevangelimann,IMDB
570. The Playroom,['Julia Dyer'],2012,NOT RATED,null,['Drama'],"Four children in their attic hideaway make 

## Standardize age certifications (Bronze → Bronze Std)

In [0]:
from pyspark.sql import functions as F

rating_map = {"G":"G","PG":"PG","PG-13":"PG-13","R":"R","NC-17":"NC-17",
              "TV-Y":"TV-Y","TV-Y7":"TV-Y7","TV-G":"TV-G","TV-PG":"TV-PG","TV-14":"TV-14","TV-MA":"TV-MA"}
alts = {"NR":"UNRATED","NOT RATED":"UNRATED","UNRATED":"UNRATED","N/A":"UNRATED","NA":"UNRATED",
        "TVY":"TV-Y","TVY7":"TV-Y7","TVG":"TV-G","TVPG":"TV-PG","TV14":"TV-14","TVMA":"TV-MA",
        "U":"G","A":"R","UA":"PG-13","12A":"PG-13","12":"PG","15":"R","18":"NC-17"}

@F.udf("string")
def normalize_cert(s):
    if s is None: return "UNRATED"
    x = s.strip().upper().split("(")[0].strip()
    if x in rating_map: return rating_map[x]
    if x in alts: return alts[x]
    return "UNRATED"

netflix_std = spark.table("workspace.default.netflix_bronze") \
    .withColumn("age_certification", normalize_cert(F.col("rating")))

imdb_std = spark.table("workspace.default.imdb_bronze") \
    .withColumn("age_certification", normalize_cert(F.col("rating")))

netflix_std.write.format("delta").mode("overwrite").saveAsTable("workspace.default.netflix_bronze_std")
imdb_std.write.format("delta").mode("overwrite").saveAsTable("workspace.default.imdb_bronze_std")
print("standardized: netflix_bronze_std, imdb_bronze_std")


standardized: netflix_bronze_std, imdb_bronze_std


## Conformed table (Netflix + IMDB)

In [0]:
common_cols = [
    "source_system","content_type","title","director","release_year",
    "age_certification","duration","genres","description","norm_title"
]

def add_missing(df, cols):
    for c in cols:
        if c not in df.columns:
            df = df.withColumn(c, F.lit(None).cast("string"))
    return df.select(*cols)

net_c  = add_missing(netflix_std, common_cols)
imdb_c = add_missing(imdb_std, common_cols)

movies_conformed = (
    net_c.unionByName(imdb_c, allowMissingColumns=True)
         .withColumn(
             "conformed_id",
             F.sha2(F.concat_ws("||",
                                F.col("source_system"),
                                F.col("norm_title"),
                                F.col("release_year").cast("string")), 256)
         )
         .dropDuplicates(["norm_title","release_year","source_system"])
)

movies_conformed.write.format("delta").mode("overwrite").saveAsTable("workspace.default.movies_conformed")
print("workspace.default.movies_conformed")

workspace.default.movies_conformed


## Analytics
### a) Number of titles released per year

In [0]:
titles_per_year = (
    spark.table("workspace.default.movies_conformed")
         .filter(F.col("release_year").isNotNull())
         .groupBy("release_year")
         .agg(F.countDistinct("conformed_id").alias("title_count"))
         .orderBy("release_year")
)
titles_per_year.write.format("delta").mode("overwrite").saveAsTable("workspace.default.titles_per_year")
display(titles_per_year)
print("workspace.default.titles_per_year")

release_year,title_count
0,1
4,1
6,1
12,1
13,1
14,5
15,2
19,1
24,1
25,2


workspace.default.titles_per_year


### b) Top directors by content production

In [0]:
top_directors = (
    spark.table("workspace.default.movies_conformed")
         .withColumn("director_exploded", F.explode_outer(F.split(F.col("director"), r"\s*,\s*")))
         .filter(F.col("director_exploded").isNotNull() & (F.length(F.col("director_exploded")) > 0))
         .groupBy("director_exploded")
         .agg(F.countDistinct("conformed_id").alias("title_count"))
         .orderBy(F.desc("title_count"), F.col("director_exploded").asc())
)
top_directors.write.format("delta").mode("overwrite").saveAsTable("workspace.default.top_directors")
display(top_directors.limit(20))
print("workspace.default.top_directors")

director_exploded,title_count
USA'],11122
California,10827
USA (Studio)'],3965
Los Angeles,3521
England,2873
New York,2079
Hollywood,1520
UK'],1268
Canada'],1195
Italy'],1054


workspace.default.top_directors


### c) Most common age certifications

In [0]:
age_certification_counts = (
    spark.table("workspace.default.movies_conformed")
         .groupBy("age_certification")
         .agg(F.countDistinct("conformed_id").alias("title_count"))
         .orderBy(F.desc("title_count"))
)
age_certification_counts.write.format("delta").mode("overwrite").saveAsTable("workspace.default.age_certification_counts")
display(age_certification_counts)
print("workspace.default.age_certification_counts")

age_certification,title_count
UNRATED,108414
R,11869
PG-13,4445
PG,3996
TV-MA,1214
G,1020
TV-14,699
TV-PG,404
TV-G,168
TV-Y7,125


workspace.default.age_certification_counts
